# LIDC-IDRI — Kaggle external pilot: step-by-step (v2)

**Purpose:** explore frozen IQ-OTH/NCCD 3-class model on an *independent, binary nodule-anchored* LIDC-IDRI cohort. This is **not** matched 3-class external validation or independently verified biopsy-to-segmentation mapping. No retraining or retuning.

**Before running:** Kaggle Notebook Settings: **GPU T4**, **Internet ON**; add both input archives as Kaggle datasets (`LIDC_IDRI_Two_Case_Pilot_Audit.zip` and `GHDNN_FROZEN_MODEL.zip`). Kaggle might unpack ZIP files: the setup cell supports either case.

**Execute cells sequentially. Do not click Run All.** Cells 1–5: install, setup, metadata/size preview, first patient smoke test, next 4 patients, save results. After verifying a batch, optional later cell advances by 5 patients.

**Cohort policy:** the two already inspected pilot patients (`LIDC-IDRI-0183`, `LIDC-IDRI-0138`) are excluded from the extended exploratory cohort (54 remaining candidates). Inferred nodule-number concordance remains unverified clinically. Patient aggregation, not slice counting.

In [ ]:
%pip -q install -U idc-index pydicom "scikit-learn==1.6.1"

In [ ]:
# CELL 2: Locate Kaggle Inputs, whether ZIP archives stayed zipped or were auto-extracted.
from pathlib import Path
import sys, zipfile, shutil, io, json
import pandas as pd
import torch

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/lidc_external_kit_v2')
WORK.mkdir(parents=True, exist_ok=True)

def find_exact(name):
    return list(INPUT.rglob(name))

kit_archives=find_exact('LIDC_IDRI_Two_Case_Pilot_Audit.zip')
model_archives=find_exact('GHDNN_FROZEN_MODEL.zip')

script=WORK/'full_external_kaggle.py'
cohort=WORK/'candidate56_concordant_nodule1.csv'

if kit_archives:
    with zipfile.ZipFile(kit_archives[0]) as z:
        for src,dst in [('LIDC_Two_Case_Pilot/full_external_kaggle.py',script),
                        ('LIDC_Two_Case_Pilot/candidate56_concordant_nodule1.csv',cohort)]:
            dst.write_bytes(z.read(src))
else:
    scripts=find_exact('full_external_kaggle.py')
    lists=find_exact('candidate56_concordant_nodule1.csv')
    assert scripts and lists, 'Kit not found: add LIDC_IDRI_Two_Case_Pilot_Audit.zip in Add Input'
    shutil.copy2(scripts[0],script)
    shutil.copy2(lists[0],cohort)

if model_archives:
    frozen=model_archives[0]
else:
    weights=find_exact('ghdnn_frozen_pipeline.joblib')
    assert weights, 'Frozen model not found: add GHDNN_FROZEN_MODEL.zip in Add Input'
    frozen=WORK/'GHDNN_FROZEN_MODEL_REPACKED.zip'
    with zipfile.ZipFile(frozen,'w',compression=zipfile.ZIP_STORED) as z:
        z.write(weights[0],arcname='ghdnn_frozen_pipeline.joblib')
        for aux in ('manifest.json','external_evaluation_protocol.txt'):
            paths=find_exact(aux)
            if paths:z.write(paths[0],arcname=aux)

# Safety fix: `startswith("Nodule 1")` also matches Nodule 10, Nodule 11, etc.
# Require a word boundary after the 1 to avoid counting incorrect nodules.
source=script.read_text(encoding='utf-8')
needle="if not segments[k].startswith('Nodule 1'):continue"
assert needle in source, 'Unexpected script version: review nodule-label selection before execution'
source=source.replace('import argparse,collections,hashlib,io,json,os,time,traceback,zipfile',
                      'import argparse,collections,hashlib,io,json,os,time,traceback,zipfile,re')
source=source.replace(needle, "if not re.match(r'^Nodule"+chr(92)+"s+1"+chr(92)+"b',segments[k],flags=re.I):continue")
script.write_text(source,encoding='utf-8')

# Exclude the 2 previously inspected pilot patients from the extended batch cohort.
original=pd.read_csv(cohort)
assert len(original)==56, f'Unexpected cohort size: {len(original)}'
exclude={'LIDC-IDRI-0183','LIDC-IDRI-0138'}
extended=original[~original['Patient ID'].isin(exclude)].reset_index(drop=True)
assert len(extended)==54, 'Check candidate list: expected 54 after excluding pilot cases'
cohort54=WORK/'candidate54_unseen_only.csv'
extended.to_csv(cohort54,index=False)

print('GPU:',torch.cuda.is_available(),torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('SCRIPT: FOUND',script,'bytes:',script.stat().st_size)
print('FROZEN MODEL: FOUND',frozen,'size_MB:',round(frozen.stat().st_size/1e6,2))
print('EXTENDED COHORT:',len(extended),'patients excluding 0183 and 0138')
print('FIRST 5 IDs:',extended['Patient ID'].head(5).tolist())
print('READY TO PREVIEW DATA SIZE')

In [ ]:
# CELL 3: Metadata-only dry run. No CT images downloaded here.
from idc_index import IDCClient
client=IDCClient.client()
preview=extended.iloc[:5]
pids=preview['Patient ID'].tolist()
quoted=','.join("'"+p.replace("'","''")+"'" for p in pids)
query=f"""SELECT PatientID, SeriesInstanceUID, Modality, series_size_MB
FROM index WHERE collection_id='lidc_idri' AND PatientID IN ({quoted})
AND ((Modality='CT' AND analysis_result_id IS NULL)
  OR (Modality='SEG' AND analysis_result_id='dicom_lidc_idri_nodules'))"""
plan=client.sql_query(query)
assert not plan.empty, 'No data returned: check Internet and current IDC index'
summary=plan.groupby(['PatientID','Modality'],dropna=False).agg(
    series=('SeriesInstanceUID','size'),size_MB=('series_size_MB','sum'))
print(summary.to_string())
print('TOTAL SIZE FIRST 5 (MB):',round(float(plan.series_size_MB.sum()),1))
print('NO IMAGES DOWNLOADED IN THIS CELL')

In [ ]:
# CELL 4: Pilot 1 patient. Download and inference can take time; outputs are saved as completed.
import subprocess
command=[sys.executable,str(script),'--candidates',str(cohort54),
         '--frozen-zip',str(frozen),'--start','0','--batch','1']
print('RUNNING ONE PATIENT:',extended.iloc[0]['Patient ID'],flush=True)
subprocess.run(command,check=True)

In [ ]:
# CELL 5: First batch remaining 4 patients (indices 1..4)
command=[sys.executable,str(script),'--candidates',str(cohort54),
         '--frozen-zip',str(frozen),'--start','1','--batch','4']
print('RUNNING FOUR MORE PATIENTS',flush=True)
subprocess.run(command,check=True)

In [ ]:
# CELL 6: View results and ZIP small outputs (CSV/JSON only; no huge DICOM folders).
from pathlib import Path
import zipfile
out=Path('/kaggle/working/LIDC_EXTERNAL')
assert out.is_dir(),'No results directory yet; finish evaluation cells first'
res=out/'patient_results.csv'
if res.exists():
    df=pd.read_csv(res)
    print('PATIENTS TESTED:',len(df))
    print(df[['patient_id','truth','prediction','n_matched_annotated_slices','correct']].to_string(index=False))
else:
    print('No successful patient predictions yet — inspect exclusions JSON files below')
for e in sorted(out.glob('exclusions_*.json')):
    print('EXCLUSIONS',e.name,':',e.read_text()[:2000])
archive=Path('/kaggle/working/LIDC_EXTERNAL_RESULTS.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.csv','.json','.txt'}:
            z.write(p,arcname=str(p.relative_to(out)))
print('DOWNLOAD THIS ZIP:',archive)

## Next batches (optional — after you have reviewed/saved the first ZIP)

For patients 6–10, set `RUN_NEXT_BATCH = True` and keep `START = 5`. For the next group change `START = 10`, then 15, 20, ... 50. There are 54 candidates total, so the last batch beginning at 50 contains 4. **After each batch rerun the previous export cell and download the new ZIP.** Keep the Kaggle session alive between batches. If you restart a fresh session, restore the previous output CSV first; otherwise earlier predictions will not automatically be retained.

In [ ]:
# OPTIONAL CELL 7: disabled by default — does NOT run until you change RUN_NEXT_BATCH to True.
RUN_NEXT_BATCH=False
START=5
if RUN_NEXT_BATCH:
    assert START in range(5,54,5), 'START must be 5,10,15,...50'
    command=[sys.executable,str(script),'--candidates',str(cohort54),
             '--frozen-zip',str(frozen),'--start',str(START),'--batch','5']
    print('RUNNING NEXT BATCH FROM INDEX',START,flush=True)
    subprocess.run(command,check=True)
    print('FINISHED: rerun Cell 6 to archive results')
else:
    print('No extra batch started. Set RUN_NEXT_BATCH=True only after downloading first results.')

### Interpretation limits

- This is an **exploratory** LIDC-IDRI 2-class nodule-anchored transfer study using a **3-class full-slice model**. It is not the matching 3-class external validation missing from the manuscript.
- IDs refer to de-identified patients, but exact association between a pathology-reported "Nodule 1" and a DICOM segment labeled "Nodule 1" remains **provisional**, not verified by independent radiology review.
- Keep the Normal predictions as errors/out-of-task outputs, do not remove them post-hoc.
- Compare at the **patient** level, not by treating multiple slices as independent patients.
- Evaluate package errors, exclusions and annotation coverage before reporting any external performance.